# YTSeg ASR Word Timestamp Export

Use this notebook in Google Colab to export YTSeg audio samples into ASR transcripts with word timestamps for chaptering evaluation.

Outputs are written to Google Drive so the job can resume after a Colab disconnect.

## Cell 1 - Install Dependencies

In [ ]:
!pip install -q datasets faster-whisper huggingface_hub soundfile pandas tqdm

## Cell 2 - Optional HF Token, Google Drive, Output Folders

In [ ]:

from pathlib import Path
import os

from google.colab import drive
from huggingface_hub import login

# Optional. In Colab, set HF_TOKEN in Secrets if Hug Face asks for auth.
hf_token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    hf_token = hf_token or userdata.get("HF_TOKEN")
except Exception:
    pass

if hf_token:
    login(token=hf_token)

drive.mount("/content/drive")

OUTPUT_ROOT = Path("/content/drive/MyDrive/chaptering_eval/ytseg")
EXPECTED_ROOT = OUTPUT_ROOT / "expected"
TRANSCRIPT_ROOT = OUTPUT_ROOT / "transcripts"
SPLIT_NAMES = ("dev", "holdout")
EXPECTED_SPLIT_DIRS = {split: EXPECTED_ROOT / split for split in SPLIT_NAMES}
TRANSCRIPT_SPLIT_DIRS = {split: TRANSCRIPT_ROOT / split for split in SPLIT_NAMES}
TMP_AUDIO_DIR = Path("/content/ytseg_tmp_audio")

for path in [
    OUTPUT_ROOT,
    EXPECTED_ROOT,
    TRANSCRIPT_ROOT,
    TMP_AUDIO_DIR,
    *EXPECTED_SPLIT_DIRS.values(),
    *TRANSCRIPT_SPLIT_DIRS.values(),
]:
    path.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = OUTPUT_ROOT / "manifest.csv"
METADATA_PATH = OUTPUT_ROOT / "selected_ytseg_metadata.json"
ERRORS_PATH = OUTPUT_ROOT / "errors.jsonl"
VALIDATION_REPORT_PATH = OUTPUT_ROOT / "validation_report.json"

print("Output root:", OUTPUT_ROOT)
print("Expected root:", EXPECTED_ROOT)
print("Transcript root:", TRANSCRIPT_ROOT)



## Cell 3 - Select Candidate Video IDs

This cell selects YTSeg videos automatically from the `text` split. It filters by duration/chapter count, samples deterministically with a fixed seed, limits repeated channels, and creates a fixed 70/30 `dev`/`holdout` split.

Use `RUN_IDS = DEV_IDS[:2]` for a smoke run, then switch to all IDs.


In [ ]:

from collections import Counter, defaultdict
import random

from datasets import load_dataset

SEED = 42
TARGET_TOTAL = 500
DEV_RATIO = 0.70
MAX_PER_CHANNEL = 3
MIN_DURATION_SECONDS = 6 * 60
MAX_DURATION_SECONDS = 30 * 60
MIN_CHAPTERS = 4
MAX_CHAPTERS = 16

rng = random.Random(SEED)


def duration_bucket(duration_seconds: float) -> str:
    if duration_seconds < 10 * 60:
        return "06_10"
    if duration_seconds < 15 * 60:
        return "10_15"
    if duration_seconds < 20 * 60:
        return "15_20"
    return "20_30"


def chapter_bucket(chapter_count: int) -> str:
    if chapter_count <= 6:
        return "04_06"
    if chapter_count <= 10:
        return "07_10"
    return "11_16"


def candidate_from_row(row: dict) -> dict | None:
    duration_seconds = float(row.get("duration") or 0.0)
    chapter_timestamps = list(row.get("chapter_timestamps") or [])
    chapter_count = len(chapter_timestamps)
    channel_id = row.get("channel_id") or ""
    speaker_category = row.get("speaker_category") or "unknown"

    if not (MIN_DURATION_SECONDS <= duration_seconds <= MAX_DURATION_SECONDS):
        return None
    if not (MIN_CHAPTERS <= chapter_count <= MAX_CHAPTERS):
        return None
    if not channel_id:
        return None

    return {
        "video_id": row["video_id"],
        "duration_seconds": duration_seconds,
        "chapter_count": chapter_count,
        "channel_id": channel_id,
        "speaker_category": speaker_category,
        "bucket": (
            duration_bucket(duration_seconds),
            chapter_bucket(chapter_count),
            speaker_category,
        ),
    }


def select_balanced_candidates(candidates: list[dict], target_total: int) -> list[dict]:
    shuffled = list(candidates)
    rng.shuffle(shuffled)

    by_bucket = defaultdict(list)
    for candidate in shuffled:
        by_bucket[candidate["bucket"]].append(candidate)

    selected = []
    channel_counts = Counter()
    bucket_order = sorted(by_bucket)

    while len(selected) < target_total:
        made_progress = False
        for bucket in bucket_order:
            items = by_bucket[bucket]
            while items:
                candidate = items.pop()
                channel_id = candidate["channel_id"]
                if channel_counts[channel_id] >= MAX_PER_CHANNEL:
                    continue

                selected.append(candidate)
                channel_counts[channel_id] += 1
                made_progress = True
                break

            if len(selected) >= target_total:
                break

        if not made_progress:
            break

    if len(selected) < target_total:
        raise ValueError(
            f"Only selected {len(selected)} videos; requested {target_total}. "
            "Relax filters or channel cap."
        )

    rng.shuffle(selected)
    return selected


def print_split_summary(name: str, rows: list[dict]) -> None:
    durations = [row["duration_seconds"] for row in rows]
    chapter_counts = [row["chapter_count"] for row in rows]
    print(
        name,
        "count", len(rows),
        "duration_avg", round(sum(durations) / len(durations), 1),
        "chapter_avg", round(sum(chapter_counts) / len(chapter_counts), 1),
        "speaker", dict(Counter(row["speaker_category"] for row in rows).most_common()),
    )


print("Loading YTSeg text split for candidate selection...")
text_ds = load_dataset("retkowski/ytseg", "text", split="test")
all_candidates = [candidate for row in text_ds if (candidate := candidate_from_row(row))]
selected_candidates = select_balanced_candidates(all_candidates, TARGET_TOTAL)

dev_count = int(len(selected_candidates) * DEV_RATIO)
dev_candidates = selected_candidates[:dev_count]
holdout_candidates = selected_candidates[dev_count:]

DEV_IDS = [candidate["video_id"] for candidate in dev_candidates]
HOLDOUT_IDS = [candidate["video_id"] for candidate in holdout_candidates]
SPLIT_BY_ID = {video_id: "dev" for video_id in DEV_IDS}
SPLIT_BY_ID.update({video_id: "holdout" for video_id in HOLDOUT_IDS})

# First run recommendation: RUN_IDS = DEV_IDS[:2]
# Full run: RUN_IDS = DEV_IDS + HOLDOUT_IDS
RUN_IDS = DEV_IDS + HOLDOUT_IDS
RUN_ID_SET = set(RUN_IDS)

print("candidate pool", len(all_candidates))
print_split_summary("dev", dev_candidates)
print_split_summary("holdout", holdout_candidates)
print("dev", len(DEV_IDS), "holdout", len(HOLDOUT_IDS), "run", len(RUN_IDS))


## Cell 4 - Validate Candidate IDs Exist In YTSeg Text Metadata

In [ ]:
from datasets import load_dataset


def scan_video_ids(dataset_name: str, config_name: str, split: str, wanted_ids: set[str]) -> set[str]:
    dataset = load_dataset(dataset_name, config_name, split=split)
    found_ids = set()
    for row in dataset:
        video_id = row.get("video_id")
        if video_id in wanted_ids:
            found_ids.add(video_id)
        if found_ids == wanted_ids:
            break
    return found_ids


duplicate_ids = sorted({video_id for video_id in RUN_IDS if RUN_IDS.count(video_id) > 1})
if duplicate_ids:
    raise ValueError(f"Duplicate RUN_IDS: {duplicate_ids}")

# Keep this check metadata-only. Do not load the YTSeg audio config here;
# it is large and can exceed Colab disk limits. The ASR cell streams audio.
print("Checking text split...")
text_found_ids = scan_video_ids("retkowski/ytseg", "text", "test", RUN_ID_SET)
missing_text_ids = sorted(RUN_ID_SET - text_found_ids)
if missing_text_ids:
    raise ValueError(f"Missing IDs in YTSeg text/test: {missing_text_ids}")

print("All candidate IDs exist in text/test.")
print("count", len(RUN_ID_SET))

## Cell 5 - Load YTSeg Data, Export Expected Chapters And Manifest

In [ ]:
import csv
import json
from datetime import datetime, timezone

from datasets import load_dataset


def atomic_write_json(path: Path, payload: dict | list) -> None:
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    tmp_path.replace(path)


def parse_timestamp(value) -> float:
    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip()
    if not text:
        return 0.0

    if ":" not in text:
        return float(text)

    parts = [float(part) for part in text.split(":")]
    seconds = 0.0
    for part in parts:
        seconds = seconds * 60 + part
    return seconds


def normalize_chapter_starts(raw_timestamps, duration_seconds: float) -> list[float]:
    starts = [parse_timestamp(value) for value in (raw_timestamps or [])]
    starts = sorted({round(start, 3) for start in starts if 0 <= start < duration_seconds})
    if not starts or starts[0] > 0.5:
        starts = [0.0, *starts]
    else:
        starts[0] = 0.0
    return starts


def expected_payload(row: dict) -> dict:
    video_id = row["video_id"]
    duration_seconds = float(row.get("duration") or 0.0)
    starts = normalize_chapter_starts(row.get("chapter_timestamps") or [], duration_seconds)
    titles = list(row.get("chapter_titles") or [])
    chapters = []
    for index, start_time in enumerate(starts):
        title = titles[index] if index < len(titles) else None
        chapters.append(
            {
                "index": index + 1,
                "startTime": round(float(start_time), 3),
                "title": title,
            }
        )

    return {
        "videoId": video_id,
        "source": "ytseg",
        "split": SPLIT_BY_ID.get(video_id),
        "durationSeconds": round(duration_seconds, 3),
        "speakerCategory": row.get("speaker_category"),
        "channelId": row.get("channel_id"),
        "chapters": chapters,
    }


print("Loading YTSeg text split...")
text_ds = load_dataset("retkowski/ytseg", "text", split="test")
text_rows_by_id = {row["video_id"]: row for row in text_ds if row["video_id"] in RUN_ID_SET}
missing_text_ids = sorted(RUN_ID_SET - set(text_rows_by_id))
if missing_text_ids:
    raise ValueError(f"Missing IDs in text split: {missing_text_ids}")

manifest_rows = []
metadata_rows = []
for video_id in RUN_IDS:
    row = text_rows_by_id[video_id]
    expected = expected_payload(row)
    split = SPLIT_BY_ID[video_id]
    expected_path = EXPECTED_SPLIT_DIRS[split] / f"{video_id}.json"
    transcript_path = TRANSCRIPT_SPLIT_DIRS[split] / f"{video_id}.json"
    atomic_write_json(expected_path, expected)

    status = "done" if transcript_path.exists() else "pending_asr"
    manifest_rows.append(
        {
            "video_id": video_id,
            "split": split,
            "duration_seconds": expected["durationSeconds"],
            "chapter_count": len(expected["chapters"]),
            "speaker_category": row.get("speaker_category"),
            "channel_id": row.get("channel_id"),
            "expected_path": str(expected_path),
            "transcript_path": str(transcript_path),
            "status": status,
        }
    )
    metadata_row = dict(row)
    metadata_row["selected_split"] = split
    metadata_rows.append(metadata_row)

with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(manifest_rows[0].keys()))
    writer.writeheader()
    writer.writerows(manifest_rows)

atomic_write_json(
    METADATA_PATH,
    {
        "createdAt": datetime.now(timezone.utc).isoformat(),
        "dataset": "retkowski/ytseg",
        "config": "text",
        "split": "test",
        "rows": metadata_rows,
    },
)

print("Expected chapters:", EXPECTED_ROOT)
print("Manifest:", MANIFEST_PATH)
print("Rows:", len(manifest_rows))


## Cell 5 - Load Faster-Whisper Model

Runtime should be GPU. In Colab: Runtime -> Change runtime type -> T4/A100 GPU.

In [ ]:
import torch
from faster_whisper import WhisperModel

MODEL_NAME = "large-v3-turbo"
DEVICE = "cuda"
COMPUTE_TYPE = "float16"

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Switch Colab runtime to GPU before loading the model.")

print("GPU:", torch.cuda.get_device_name(0))
model = WhisperModel(MODEL_NAME, device=DEVICE, compute_type=COMPUTE_TYPE)
print("Loaded", MODEL_NAME)

## Cell 6 - Run ASR Batch And Save Transcripts

This cell skips existing transcript JSON files, so it is safe to rerun.

In [ ]:
import csv
import json
import traceback
from datetime import datetime, timezone

import soundfile as sf
from datasets import Audio, load_dataset
from tqdm.auto import tqdm


def load_manifest() -> list[dict]:
    with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as file:
        return list(csv.DictReader(file))


def write_manifest(rows: list[dict]) -> None:
    if not rows:
        return
    with MANIFEST_PATH.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def append_error(video_id: str, error: BaseException) -> None:
    payload = {
        "videoId": video_id,
        "createdAt": datetime.now(timezone.utc).isoformat(),
        "errorType": type(error).__name__,
        "error": str(error),
        "traceback": traceback.format_exc(),
    }
    with ERRORS_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(payload, ensure_ascii=False) + "\n")


def audio_input_path(video_id: str, audio_value: dict) -> Path:
    source_path = audio_value.get("path")
    if source_path and Path(source_path).exists():
        return Path(source_path)

    audio_bytes = audio_value.get("bytes")
    if audio_bytes:
        suffix = Path(source_path or "audio.mp3").suffix or ".mp3"
        temp_path = TMP_AUDIO_DIR / f"{video_id}{suffix}"
        temp_path.write_bytes(audio_bytes)
        return temp_path

    array = audio_value["array"]
    sampling_rate = int(audio_value["sampling_rate"])
    temp_path = TMP_AUDIO_DIR / f"{video_id}.wav"
    sf.write(temp_path, array, sampling_rate)
    return temp_path


def transcribe_audio(video_id: str, audio_path: Path, expected: dict) -> dict:
    segments_iter, info = model.transcribe(
        str(audio_path),
        language="en",
        beam_size=5,
        vad_filter=True,
        word_timestamps=True,
    )

    output_segments = []
    word_index = 1
    for segment_index, segment in enumerate(segments_iter, start=1):
        segment_id = f"seg_{segment_index:06d}"
        words = []
        for word in segment.words or []:
            if word.start is None or word.end is None:
                continue
            text = (word.word or "").strip()
            if not text:
                continue
            start = round(float(word.start), 3)
            end = round(float(word.end), 3)
            if start >= end:
                continue
            words.append(
                {
                    "id": f"w_{word_index:06d}",
                    "segmentId": segment_id,
                    "start": start,
                    "end": end,
                    "text": text,
                }
            )
            word_index += 1

        output_segments.append(
            {
                "id": segment_id,
                "start": round(float(segment.start), 3),
                "end": round(float(segment.end), 3),
                "text": (segment.text or "").strip(),
                "words": words,
            }
        )

    return {
        "videoId": video_id,
        "source": "ytseg",
        "split": expected.get("split"),
        "language": getattr(info, "language", "en") or "en",
        "durationSeconds": expected["durationSeconds"],
        "asr": {
            "provider": "faster-whisper",
            "model": MODEL_NAME,
            "device": DEVICE,
            "computeType": COMPUTE_TYPE,
            "wordTimestamps": True,
        },
        "segments": output_segments,
    }


manifest_rows = load_manifest()
manifest_by_id = {row["video_id"]: row for row in manifest_rows}
pending_ids = {
    video_id
    for video_id in RUN_ID_SET
    if not Path(manifest_by_id[video_id]["transcript_path"]).exists()
}

if not pending_ids:
    print("All requested transcripts already exist. Nothing to transcribe.")
else:
    print("Streaming YTSeg audio split for pending IDs:", len(pending_ids))

# Streaming avoids downloading/caching the full audio dataset on Colab disk.
# decode=False keeps rows lightweight until a selected ID is written to tmp audio.
audio_ds = load_dataset("retkowski/ytseg", "audio", split="test", streaming=True)
audio_ds = audio_ds.cast_column("audio", Audio(decode=False))

for row in tqdm(audio_ds, desc="Stream audio rows"):
    video_id = row["video_id"]
    if video_id not in pending_ids:
        continue

    manifest_row = manifest_by_id[video_id]
    output_path = Path(manifest_row["transcript_path"])
    if output_path.exists():
        manifest_row["status"] = "done"
        write_manifest(manifest_rows)
        pending_ids.discard(video_id)
        if not pending_ids:
            break
        continue

    try:
        expected = json.loads(Path(manifest_row["expected_path"]).read_text(encoding="utf-8"))
        audio_path = audio_input_path(video_id, row["audio"])
        transcript = transcribe_audio(video_id, audio_path, expected)
        atomic_write_json(output_path, transcript)
        manifest_row["status"] = "done"
    except Exception as error:
        manifest_row["status"] = "failed"
        append_error(video_id, error)
        print("FAILED", video_id, error)
    finally:
        pending_ids.discard(video_id)
        write_manifest(manifest_rows)
        if not pending_ids:
            break

if pending_ids:
    missing_audio_ids = sorted(pending_ids)
    raise ValueError(f"Missing IDs in streamed audio split: {missing_audio_ids}")

print("Done. Transcripts:", TRANSCRIPT_ROOT)
print("Errors:", ERRORS_PATH)


## Cell 7 - Validate Outputs And Print Summary

In [ ]:
import csv
import json
from collections import Counter


def validate_expected(expected: dict) -> list[str]:
    errors = []
    duration = float(expected.get("durationSeconds") or 0.0)
    chapters = expected.get("chapters") or []
    starts = [float(chapter.get("startTime") or 0.0) for chapter in chapters]
    if duration <= 0:
        errors.append("expected duration must be positive")
    if not starts or abs(starts[0]) > 0.5:
        errors.append("expected chapters must start at 0")
    if starts != sorted(starts):
        errors.append("expected chapter starts must be sorted")
    if any(start < 0 or start >= duration for start in starts):
        errors.append("expected chapter start outside duration")
    return errors


def validate_transcript(transcript: dict, expected: dict) -> list[str]:
    errors = []
    duration = float(transcript.get("durationSeconds") or 0.0)
    expected_duration = float(expected.get("durationSeconds") or 0.0)
    segments = transcript.get("segments") or []
    if abs(duration - expected_duration) > 1.0:
        errors.append("transcript duration differs from expected duration")
    if not segments:
        errors.append("segments are empty")
        return errors
    previous_segment_start = -1.0
    word_count = 0
    previous_word_start = -1.0
    for segment in segments:
        start = float(segment.get("start") or 0.0)
        end = float(segment.get("end") or 0.0)
        if start < previous_segment_start:
            errors.append("segments are not sorted")
        if start >= end:
            errors.append("segment start must be before end")
        if end > duration + 5.0:
            errors.append("segment exceeds duration tolerance")
        previous_segment_start = start
        for word in segment.get("words") or []:
            word_start = float(word.get("start") or 0.0)
            word_end = float(word.get("end") or 0.0)
            if word_start < previous_word_start:
                errors.append("words are not sorted")
            if word_start >= word_end:
                errors.append("word start must be before end")
            previous_word_start = word_start
            word_count += 1
    if word_count == 0:
        errors.append("words are empty")
    return sorted(set(errors))


with MANIFEST_PATH.open("r", newline="", encoding="utf-8") as file:
    manifest_rows = list(csv.DictReader(file))

report = []
status_counter = Counter()
for row in manifest_rows:
    video_id = row["video_id"]
    expected_path = Path(row["expected_path"])
    transcript_path = Path(row["transcript_path"])
    expected = json.loads(expected_path.read_text(encoding="utf-8"))
    errors = validate_expected(expected)
    if transcript_path.exists():
        transcript = json.loads(transcript_path.read_text(encoding="utf-8"))
        errors.extend(validate_transcript(transcript, expected))
    else:
        errors.append("transcript missing")
    status = "valid" if not errors else "invalid"
    status_counter[status] += 1
    report.append(
        {
            "videoId": video_id,
            "split": row["split"],
            "status": status,
            "errors": sorted(set(errors)),
        }
    )

atomic_write_json(VALIDATION_REPORT_PATH, report)

print("Validation:", dict(status_counter))
print("Report:", VALIDATION_REPORT_PATH)
for item in report[:10]:
    if item["errors"]:
        print(item)